# Models

And now - this colab unveils the heart (or the brains?) of the transformers library - the models:

https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy?usp=sharing

This should run nicely on a low-cost or free T4 box.

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModeForCausalLM, TextStreamer, BitsAndBytesConfig 

In [ ]:
# Load environment variable
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [ ]:
# Instruct models
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI3 = "microsoft/Phi-3-mini-4k-instruct"
GEMMA2 = "google/gemma-2-2b-it"
QWEN2 = "Qwen/Qwen2-7B-Instruct"
MIXTRAL = "mistralai/Mixtral-8x7B-Instruct-v0.1"

In [ ]:
# Chat history
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room full of Data Scientists"}
]

In [ ]:
# Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                               # Load weights in 4-bit precision
    bnb_4bit_use_double_quant=True,                  # Quantize the weights twice
    bnb_4bit_compute_dtype=torch.bfloat16,           # Use specified datatype
    bnb_4bit_quant_type="nf4"                        # How to treat/compress down to 4-bit
)

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensor="pt").to("cuda")

In [ ]:
# Model using quantization config to load model efficiently
model = AutoModelForCausalLLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [ ]:
model

In [ ]:
# Generate output using tokenized inputs and decode it to readable format
outputs = model.generate(inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

In [ ]:
# Cleanup space
del inputs, outputs, model
torch.cuda.empty_cache()

In [ ]:
# Wrapping everything up in a single function
def generate(model_name, messages):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    streamer = TextStreamer(tokenizer)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map="auto", 
        quantization_config=quant_config
    )
    tokenized_inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)
    del tokenizer, streamer, model, inputs, outputs
    torch.cuda.empty_cache()

In [ ]:
generate(LLAMA, messages)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]
generate(GEMMA2, messages)